# 02 - AutoML Model Training

Train the recovery-risk model (P(successful recovery)) using:
1. **Baseline** - scikit-learn HistGradientBoosting (always available)
2. **H2O AutoML / AutoGluon** - heavier, optional

For production training, replace the weak-supervision labels with real outcomes:
join `payments` to `recoveries` on `recoveries.payment_id = payments.id` and use
`recovered_amount_paise > 0` as the positive class.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
from scripts.train_automl import build_dataset

X, y, feature_names = build_dataset(5000, seed=42)
print(X.shape, 'positive rate', y.mean().round(3))

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
model = HistGradientBoostingClassifier(max_iter=200).fit(X_tr, y_tr)
proba = model.predict_proba(X_te)[:, 1]
print('ROC-AUC:', round(roc_auc_score(y_te, proba), 4))

In [ ]:
# Persist in the format app.ml.risk_scorer expects
import joblib
joblib.dump(model, '../models/risk_scorer.joblib')
print('saved -> models/risk_scorer.joblib')

In [ ]:
# Optional: H2O AutoML (pip install h2o; Java 8+ required)
# from scripts.train_automl import train_h2o
# h2o_model = train_h2o(X_tr, y_tr, feature_names)

In [ ]:
# Sanity check through the runtime scorer (should report ml_model contribution)
import sys
sys.path.insert(0, '..')
from app.ml.risk_scorer import RiskScorer
scorer = RiskScorer(model_path='../models/risk_scorer.joblib')
row = X_te[0].tolist()
# scorer.score(dict(zip(feature_names, row)))  # requires model dir context